In [ ]:
import os
import json
import numpy as np
import pandas as pd
from PIL import Image
import tf_keras as keras
import google.generativeai as genai

# 1. Đọc API Key từ config.json
with open('config.json', 'r', encoding='utf-8') as f:
    config = json.load(f)

GEMINI_API_KEY = config.get("GEMINI_API_KEY", "AQ.Ab8RN6KDdBuVWoeA99pk0AZiayovvQYZM9Jk9Vyjsqp0GY9o7Q")
genai.configure(api_key=GEMINI_API_KEY)

# 2. Khởi tạo model Gemini
gemini_model = genai.GenerativeModel('gemini-1.5-flash')

# 3. Load mô hình CNN đã huấn luyện
cnn_model = keras.models.load_model('models/hardware_cnn_model.h5')

# 4. Danh sách các lớp linh kiện (theo thứ tự chữ cái giống ImageDataGenerator)
CLASS_NAMES = ['CPU', 'GPU', 'Mainboard', 'PSU', 'RAM', 'SSD']

# 5. Load file CSV cơ sở dữ liệu linh kiện
df_hardware = pd.read_csv('linh-kien-pc.csv')

print("✅ Đã load thành công Gemini API, mô hình CNN và cơ sở dữ liệu linh kiện!")

In [ ]:
def predict_hardware_image(image_path):
    """
    Hàm nhận vào đường dẫn ảnh, tiền xử lý và đưa qua mô hình CNN để phân loại linh kiện
    """
    if not os.path.exists(image_path):
        return None, 0.0
    
    # Mở và tiền xử lý ảnh về kích thước 224x224
    img = Image.open(image_path).convert('RGB')
    img_resized = img.resize((224, 224))
    img_array = np.array(img_resized) / 255.0  # Chuẩn hóa về [0, 1]
    img_array = np.expand_dims(img_array, axis=0)  # Thêm chiều batch

    # Dự đoán
    predictions = cnn_model.predict(img_array, verbose=0)
    predicted_index = np.argmax(predictions[0])
    confidence = float(predictions[0][predicted_index])
    predicted_class = CLASS_NAMES[predicted_index]

    return predicted_class, confidence

In [ ]:
def ask_hardware_assistant(user_prompt, image_path=None):
    """
    Hàm tư vấn tổng hợp: Phân loại ảnh (nếu có) + Tra cứu CSV + Hỏi đáp Gemini
    """
    context_info = ""
    detected_category = None

    # Nếu người dùng có gửi kèm ảnh
    if image_path and os.path.exists(image_path):
        category, conf = predict_hardware_image(image_path)
        detected_category = category
        context_info += f"\n[Hệ thống CNN nhận diện ảnh người dùng gửi là: {category} với độ tin cậy {conf*100:.1f}%]\n"
        
        # Tra cứu các sản phẩm thuộc nhóm này trong file CSV
        matched_items = df_hardware[df_hardware['Category'].str.upper() == category.upper()]
        if not matched_items.empty:
            context_info += f"Các sản phẩm mẫu thuộc nhóm {category} trong cơ sở dữ liệu:\n"
            context_info += matched_items.to_string(index=False) + "\n"

    # Xây dựng System Prompt cho Gemini
    system_instruction = f"""
    Bạn là một chuyên gia tư vấn phần cứng máy tính (PC Hardware Assistant) thông minh và thân thiện.
    Nhiệm vụ của bạn:
    1. Trả lời câu hỏi tư vấn cấu hình, tương thích linh kiện, giá cả, hiệu năng PC.
    2. Nếu có thông tin nhận diện linh kiện từ ảnh hoặc dữ liệu CSV bên dưới, hãy tận dụng nó để tư vấn chi tiết cho người dùng.
    3. Trả lời bằng tiếng Việt chuyên nghiệp, ngắn gọn, dễ hiểu và đưa ra giải pháp cụ thể.

    Ngữ cảnh dữ liệu bổ sung (nếu có):
    {context_info}
    """

    full_prompt = f"{system_instruction}\n\nCâu hỏi của người dùng: {user_prompt}"

    # Gọi Gemini sinh phản hồi
    response = gemini_model.generate_content(full_prompt)
    return response.text, detected_category

In [ ]:
print("=" * 60)
print("🤖 CỬA HÀNG TƯ VẤN LINH KIỆN PC SẴN SÀNG! (Gõ 'exit' để thoát)")
print("=" * 60)

# Chạy thử nghiệm chatbot
while True:
    user_input = input("\n👤 Bạn: ")
    if user_input.lower().strip() in ['exit', 'quit', 'thoát']:
        print("🤖 Chatbot: Tạm biệt bạn! Chúc bạn chọn được cấu hình PC ưng ý.")
        break

    # Nhập đường dẫn ảnh (nếu có, nhấn Enter để bỏ qua)
    img_path = input("🖼️ Nhập đường dẫn ảnh linh kiện (để trống nếu không có): ").strip()
    img_path = img_path if img_path != "" else None

    print("\n⏳ Chatbot đang suy nghĩ...")
    try:
        reply, category = ask_hardware_assistant(user_input, img_path)
        if category:
            print(f"🔍 [CNN Detected]: Linh kiện trong ảnh là {category}")
        print(f"\n🤖 Chatbot:\n{reply}")
    except Exception as e:
        print(f"❌ Lỗi: {e}")